# Trace Count v5.3: Attention, Ablation, and Causal Patching

This notebook **does not retrain v5**. It restores the corrected `trace_indices=true` v5 checkpoint and tests the paper's working hypothesis:

- **Non-thinking:** broad aggregation into one final-answer state.
- **Thinking:** indexed, item-specific retrieval followed by progress/stop updates and trace-mediated final readout.

The notebook separates descriptive attention from causal evidence using head masks, free-running ablations, clean-to-corrupt head-output patching, residual patching, prompt/trace conflicts, and progress-state transplants.

## 1. Mount Google Drive first

In [ ]:
from pathlib import Path

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Local runtime: Google Drive mount skipped.')

## 2. Repository and environment

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git'
if IN_COLAB:
    REPO_DIR = Path('/content/Synthetic_CoT_NiaH_Count')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = Path.cwd().resolve()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('repo =', REPO_DIR)
print('python =', sys.executable)

## 3. Restore the completed v5 checkpoint from Drive

In [ ]:
import json, shutil

DRIVE_RESULTS_ROOT = Path('/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results') if IN_COLAB else REPO_DIR / 'colab_results'
SOURCE_RUN_OVERRIDE = None  # Set to an exact completed v5 run if automatic discovery is ambiguous.
LOCAL_RUN = Path('/content/v5_explicit_switch_source') if IN_COLAB else REPO_DIR / 'colab_results' / 'v5_explicit_switch'

def is_v5_run(path):
    return all((path / rel).exists() for rel in ['config.json', 'vocab.json', 'checkpoints/final.pt'])

if SOURCE_RUN_OVERRIDE is not None:
    source_run = Path(SOURCE_RUN_OVERRIDE)
else:
    direct = DRIVE_RESULTS_ROOT / 'v5_explicit_switch'
    candidates = [direct] + [p.parent for p in DRIVE_RESULTS_ROOT.rglob('config.json')]
    valid = [p for p in candidates if is_v5_run(p)]
    if not valid:
        raise FileNotFoundError(f'No completed v5 run under {DRIVE_RESULTS_ROOT}. Expected config.json, vocab.json, checkpoints/final.pt.')
    source_run = max(valid, key=lambda p: (p / 'checkpoints' / 'final.pt').stat().st_mtime)

cfg = json.loads((source_run / 'config.json').read_text(encoding='utf-8'))
if not cfg.get('trace_indices', False):
    raise ValueError('Selected checkpoint is legacy marker-only v5. v5.3 requires trace_indices=true.')
if IN_COLAB:
    if LOCAL_RUN.exists():
        shutil.rmtree(LOCAL_RUN)
    shutil.copytree(source_run, LOCAL_RUN, ignore=shutil.ignore_patterns('v5_3_mechanism_causal'))
else:
    LOCAL_RUN = source_run
print('source_run =', source_run)
print('local_run =', LOCAL_RUN)
print({'trace_indices': cfg['trace_indices'], 'format_version': cfg.get('format_version'), 'model': cfg['model'], 'train': cfg['train']})

## 4. Runtime settings

`debug` is a fast pipeline check. `main` gives stable head rankings and causal averages. Patching is the slowest stage because each intervention requires another forward pass.

In [ ]:
import torch

PRESET = 'main'  # 'debug' or 'main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SETTINGS = {
    'debug': dict(attention=2, ablation=1, generation=1, patch=1, conflict=1),
    'main': dict(attention=30, ablation=10, generation=3, patch=10, conflict=10),
}[PRESET]
print({'PRESET': PRESET, 'DEVICE': DEVICE, **SETTINGS})

## 5. Run v5.3 mechanism diagnostics

The output tables retain per-example rows and aggregated summaries. `normalized_recovery = (patched - corrupt) / (clean - corrupt)`: 0 means no recovery; 1 means full clean recovery; values outside `[0,1]` are intentionally not clipped.

In [ ]:
from synthetic_counting_extensions.v5_3_mechanism_causal import run_v5_3_mechanism_causal

outputs = run_v5_3_mechanism_causal(
    LOCAL_RUN,
    attention_examples_per_count=SETTINGS['attention'],
    ablation_examples_per_count=SETTINGS['ablation'],
    generation_examples_per_count=SETTINGS['generation'],
    patch_examples_per_count=SETTINGS['patch'],
    conflict_examples_per_count=SETTINGS['conflict'],
    device=DEVICE,
)
OUT_DIR = LOCAL_RUN / 'v5_3_mechanism_causal'
print('OUT_DIR =', OUT_DIR)

## 6. Inspect head signatures

- **Direct broad score:** prompt-needle mass × normalized entropy over needles. It is high only when a head both attends to needles and distributes that attention broadly.
- **Thinking targeted score:** raw attention mass from `<I_k>` to prompt needle k, plus top-1 and diagonal dominance.
- **Thinking readout:** attention from `</Think>` (the state predicting `<C_n>`) to all trace markers.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

display(Image(filename=str(OUT_DIR / 'figures' / 'attention_mechanism_comparison.png')))
attn = outputs['attention_head_summary']
for mode, query, metric in [
    ('nonthinking', 'final_count_query', 'broad_aggregation_score'),
    ('thinking', 'trace_marker_query', 'correct_prompt_needle_mass'),
    ('thinking', 'final_count_query', 'trace_markers_mass'),
]:
    display(Markdown(f'**{mode} / {query}, ranked by `{metric}`**'))
    display(attn[(attn['mode']==mode) & (attn.query_kind==query)].sort_values(metric, ascending=False).head(8))

## 7. Ablation: necessity at three stages

Teacher-forced masks measure logit-margin drops at the exact site. Free-running masks test whether those local effects survive autoregressive generation. Random groups are matched by number of masked heads; whole-layer and all-head masks are stronger controls.

In [ ]:
for name in ['nonthinking_ablation.png','thinking_retrieval_ablation.png','thinking_readout_ablation.png','behavioral_head_ablation.png']:
    display(Image(filename=str(OUT_DIR / 'figures' / name)))
display(outputs['head_ablation_summary'].sort_values(['mode','drop_count_margin'], ascending=[True,False]).head(30))
display(outputs['behavioral_ablation_summary'].sort_values(['mode','final_accuracy']).head(30))

## 8. Patching: sufficiency and stage ordering

`retrieval_identity` changes only one prompt marker identity and patches clean head output at `<I_k>` into the corrupt run. Direct count patching deletes one needle but preserves prompt length and answer position. Thinking count-readout patching compares different trace lengths, so `position_matched=0` and must be interpreted with the learned-position confound. Residual patching scans where clean information becomes sufficient.

In [ ]:
for name in ['patching_retrieval.png', 'patching_nonthinking_readout.png', 'patching_thinking_readout.png']:
    display(Image(filename=str(OUT_DIR / 'figures' / name)))
patch = outputs['patching_summary'].sort_values(['experiment','normalized_recovery'], ascending=[True,False])
display(patch)

## 9. Progress and final aggregation

The conflict grid forces a trace of length `m` after a prompt with count `n`; the model then predicts the final count. The transplant test replaces the residual state after trace marker `k` with a state from `k-1` or `k+1`, and measures whether the next-index logit moves in the donor direction. Because k±1 states occupy different absolute positions, the same-progress/same-position donor is an essential control.

In [ ]:
display(Image(filename=str(OUT_DIR / 'figures' / 'trace_conflict_pred_count.png')))
display(Image(filename=str(OUT_DIR / 'figures' / 'progress_state_transplant.png')))
display(outputs['progress_transplant_summary'])

## 10. Save the complete diagnostic bundle to Google Drive

In [ ]:
from datetime import datetime

DRIVE_SAVE_COMPLETED = False
if IN_COLAB:
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    destination = DRIVE_RESULTS_ROOT / f'v5_3_mechanism_causal_seed1234_{stamp}'
    shutil.copytree(OUT_DIR, destination)
    DRIVE_SAVE_COMPLETED = True
    print('saved to', destination)
else:
    print('Local run already lives at', OUT_DIR)

## 11. Optional: disconnect the Colab runtime after Drive save

In [ ]:
AUTO_DISCONNECT = False
if IN_COLAB and AUTO_DISCONNECT:
    if not DRIVE_SAVE_COMPLETED:
        raise RuntimeError('Refusing to disconnect before a confirmed Drive save.')
    from google.colab import runtime
    runtime.unassign()
else:
    print('Runtime left connected. Set AUTO_DISCONNECT=True only after inspecting results.')